In [18]:

import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
from sklearn.model_selection import train_test_split

# CONFIG

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4

IMAGE_DIR_1 = "/content/drive/MyDrive/Capstone Project/data/HAM10000_images_part_1"
IMAGE_DIR_2 = "/content/drive/MyDrive/Capstone Project/data/HAM10000_images_part_2"
CSV_PATH = "/content/drive/MyDrive/Capstone Project/data/HAM10000_metadata.csv"

SAVE_PATH = "models/efficientnet_skin_best.pth"

CLASSES = ["akiec","bcc","bkl","df","mel","nv","vasc"]

os.makedirs("models", exist_ok=True)

# DATASET

class SkinDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["image_id"]
        label = CLASSES.index(row["dx"])

        path1 = os.path.join(IMAGE_DIR_1, img_id + ".jpg")
        path2 = os.path.join(IMAGE_DIR_2, img_id + ".jpg")
        img_path = path1 if os.path.exists(path1) else path2

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# TRANSFORMS

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# LOAD DATA

df = pd.read_csv(CSV_PATH)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["dx"],
    random_state=42
)

train_dataset = SkinDataset(train_df, train_transform)
val_dataset = SkinDataset(val_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

# CLASS WEIGHTS

counts = torch.tensor([327,514,1099,115,1113,6705,142], dtype=torch.float)

weights = counts.sum() / counts
weights = weights / weights.mean()

weights = weights.to(DEVICE)


Using device: cuda


In [19]:

# MODEL

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Phase 1: Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 7)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# LR SCHEDULER
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

best_acc = 0

# TRAIN LOOP

for epoch in range(EPOCHS):

    # UNFREEZE AFTER 5 EPOCHS
    if epoch == 5:
        print("\n Unfreezing backbone for fine-tuning...\n")
        for param in model.features.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss = train_loss / len(train_loader.dataset)

    # VALIDATION

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {acc:.4f}")

    scheduler.step()

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(" Model Saved")

print("\n Training Complete")
print("Best Accuracy:", best_acc)


Epoch 1/20
Train Loss: 1.8641
Val Accuracy: 0.4743
 Model Saved

Epoch 2/20
Train Loss: 1.7250
Val Accuracy: 0.5307
 Model Saved

Epoch 3/20
Train Loss: 1.6301
Val Accuracy: 0.5522
 Model Saved

Epoch 4/20
Train Loss: 1.5645
Val Accuracy: 0.5482

Epoch 5/20
Train Loss: 1.5333
Val Accuracy: 0.5392

 Unfreezing backbone for fine-tuning...


Epoch 6/20
Train Loss: 1.4589
Val Accuracy: 0.5771
 Model Saved

Epoch 7/20
Train Loss: 1.3530
Val Accuracy: 0.5776
 Model Saved

Epoch 8/20
Train Loss: 1.2622
Val Accuracy: 0.5931
 Model Saved

Epoch 9/20
Train Loss: 1.1724
Val Accuracy: 0.6026
 Model Saved

Epoch 10/20
Train Loss: 1.1144
Val Accuracy: 0.6430
 Model Saved

Epoch 11/20
Train Loss: 1.0582
Val Accuracy: 0.6400

Epoch 12/20
Train Loss: 1.0214
Val Accuracy: 0.6470
 Model Saved

Epoch 13/20
Train Loss: 0.9642
Val Accuracy: 0.6660
 Model Saved

Epoch 14/20
Train Loss: 0.9359
Val Accuracy: 0.6650

Epoch 15/20
Train Loss: 0.8916
Val Accuracy: 0.6855
 Model Saved

Epoch 16/20
Train Loss: 0.85